# A/B Testing

Run the next cell once to install the packages this notebook uses. Then run the `# datascience` and `# polars` import cells. If an import still fails, restart the kernel and run from the top.


In [ ]:
# setup
# Installs this notebook's packages for the current Python.
%pip install -q \
    pyarrow \
    polars==2.1.0 \
    hvplot==0.12.2

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import warnings
warnings.simplefilter("ignore")

In [ ]:
# polars
import polars as pl
import hvplot.polars
# from IPython.display import display

## Comparing Two Samples

In [ ]:
# datascience
births = Table.read_table('baby.csv')

In [ ]:
# polars - Note: Some CSVs may require `eol_char='\r'` to load them, but we'll update the files so students won't have to worry about this.
births_pl = pl.read_csv('baby.csv', eol_char='\r')

In [ ]:
# datascience
births

In [ ]:
# polars
births_pl

In [ ]:
# datascience
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')

In [ ]:
# polars
smoking_and_birthweight_pl = births_pl.select('Maternal Smoker', 'Birth Weight')

In [ ]:
# datascience
smoking_and_birthweight.group('Maternal Smoker')

In [ ]:
# polars: Note - The order of rows returned from `.group_by` is non-deterministic. To fix, include `.sort` 
smoking_and_birthweight_pl.group_by('Maternal Smoker').len().sort('Maternal Smoker')

In [ ]:
# datascience
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

In [ ]:
# polars: Note - Only the first three arguments are essential, the rest are for styling purposes
display(
    smoking_and_birthweight_pl.hvplot.hist(
        'Birth Weight',
        by='Maternal Smoker',
        normed=True, # sets y-axis to density rather than count
        bins=10,
        alpha=0.5, # sets opacity
        color=['#002fa7', '#706700'], # blue and gold
    )
)

## Test Statistic

[Question] What values of our statistic are in favor of the alternative: positive or negative?

In [ ]:
# datascience
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

In [ ]:
# polars
means_table_pl = (
    smoking_and_birthweight_pl.group_by('Maternal Smoker').mean()
    .rename({'Birth Weight': 'Birth Weight average'}) # optional argument if we want to rename the column name
)
means_table_pl

In [ ]:
# datascience
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

In [ ]:
# polars
means_pl = means_table_pl.to_series(1) # or .get_column('Maternal Smoker')
observed_difference_pl = means_pl.item(1) - means_pl.item(0)
observed_difference_pl

In [ ]:
# datascience
def difference_of_means(table, numeric_label, category_label):
    """
    Takes: 
       - name of table
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    #table with the two relevant columns
    reduced = table.select(numeric_label, category_label)  
    
    # table containing group means
    means_table = reduced.group(category_label, np.average)
    
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
# polars
def difference_of_means_pl(df, numeric_label, category_label):
    """
    Takes: 
       - name of df
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    #table with the two relevant columns
    reduced = df.select(numeric_label, category_label)  
    
    # table containing group means
    means_df_pl = (
        reduced
        .group_by(category_label)
        .mean()
        .sort(category_label)
    )
    
    # array of group means
    means_pl = means_df_pl.get_column(numeric_label)
    
    return means_pl.item(1) - means_pl.item(0)

In [ ]:
# datascience
difference_of_means(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
difference_of_means_pl(births_pl, 'Birth Weight', 'Maternal Smoker')

## Random Permutation (Shuffling)

In [ ]:
# datascience
staff = Table().with_columns(
    'Names', make_array('Jim', 'Pam', 'Dwight', 'Michael'),
    'Ages', make_array(29, 28, 34, 41)
)

In [ ]:
# polars
staff_pl = pl.DataFrame([
    pl.Series('Names', ['Jim', 'Pam', 'Dwight', 'Michael']),
    pl.Series('Ages', [29, 28, 34, 41])
])

In [ ]:
# datascience
staff.sample()

In [ ]:
# polars
staff_pl.sample(n=staff_pl.height, with_replacement=True)

In [ ]:
# datascience
staff.sample(with_replacement = False)

In [ ]:
# polars: Note - this code is shown for comparative purposes, but in practice, we'll 
# most likely shuffle labels using `pl.col(x).shuffle()` instead
staff_pl.sample(n=staff_pl.height, shuffle=True)

In [ ]:
# datascience
staff.with_column('Shuffled', staff.sample(with_replacement = False).column(0))

In [ ]:
# polars
staff_pl.with_columns(pl.col('Names').shuffle().alias('Shuffled'))

## Simulation Under Null Hypothesis

In [ ]:
# datascience
smoking_and_birthweight

In [ ]:
# polars
smoking_and_birthweight_pl

In [ ]:
# datascience
shuffled_labels = smoking_and_birthweight.sample(with_replacement=False).column('Maternal Smoker')

In [ ]:
# polars
shuffled_labels_pl = smoking_and_birthweight_pl.select(pl.col('Maternal Smoker').shuffle()).get_column('Maternal Smoker')

In [ ]:
# datascience
original_and_shuffled = smoking_and_birthweight.with_column(
    'Shuffled Label', shuffled_labels
)

In [ ]:
# polars
original_and_shuffled_pl = smoking_and_birthweight_pl.with_columns(
    shuffled_labels_pl.alias('Shuffled Label')
)

In [ ]:
# datascience
original_and_shuffled

In [ ]:
# polars
original_and_shuffled_pl

In [ ]:
# datascience
difference_of_means(original_and_shuffled, 'Birth Weight', 'Shuffled Label')

In [ ]:
# polars
difference_of_means_pl(original_and_shuffled_pl, 'Birth Weight', 'Shuffled Label')

In [ ]:
# datascience
difference_of_means(original_and_shuffled, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
difference_of_means_pl(original_and_shuffled_pl, 'Birth Weight', 'Maternal Smoker')

## Permutation Test

In [ ]:
# datascience
def one_simulated_difference(table, numeric_label, category_label):
    """
    Takes: 
       - name of table
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(category_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(numeric_label).with_column('Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, numeric_label, 'Shuffled Label')   

In [ ]:
# polars
def one_simulated_difference_pl(df, numeric_label, category_label):
    """
    Takes: 
       - name of df
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    # array of shuffled labels
    shuffled_labels_pl = df.select(pl.col(category_label).shuffle()).get_column(category_label)
    
    # table of numerical variable and shuffled labels
    shuffled_df = df.select(numeric_label).with_columns(shuffled_labels_pl.alias('Shuffled Label'))
    
    return difference_of_means_pl(shuffled_df, numeric_label, 'Shuffled Label')

In [ ]:
# datascience
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
one_simulated_difference_pl(births_pl, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# datascience
differences = make_array()

for i in np.arange(2000):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

In [ ]:
# polars
differences_pl = []

for i in range(2000):
    new_difference = one_simulated_difference_pl(births_pl, 'Birth Weight', 'Maternal Smoker')
    differences_pl.append(new_difference)

In [ ]:
# datascience
Table().with_column('Difference Between Group Means', differences).hist()
plots.ylim(-.02, .4)
plots.scatter(observed_difference, 0, color='red', s=30);
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');

In [ ]:
# polars
print('Observed Difference:', observed_difference_pl)
(
    pl.DataFrame([pl.Series('Difference Between Group Means', differences_pl)
    ]).hvplot.hist(
        'Difference Between Group Means',
        bins=10,
        normed=True,
        title='Prediction Under the Null Hypothesis',
        ylim=(-0.02, 0.4),
    )
    * pl.DataFrame([
        pl.Series('Difference Between Group Means', [observed_difference_pl]),
        pl.Series('Frequency', [0]),
    ]).hvplot.scatter(
        x='Difference Between Group Means',
        y='Frequency',
        color='red',
        size=100,
    )
)